# Write Rates in PLATO's Format
### Import libraries

In [ ]:
import os
import sys
import pickle
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

Visualization libraries

In [ ]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
# %matplotlib qt

Global paths

In [ ]:
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']
# PyCOMET
PyCOMET = WORKSPACE + '/ML/PyCOMET/pycomet/'
if importlib.util.find_spec('pycomet') is None:
    sys.path.append(os.path.abspath(PyCOMET))
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))
# Current Working Directory
CWD = os.getcwd()

In [ ]:
# Output directory
out_dir = WORKSPACE + '/AirDatabase/Run0D/database/kinetics/SurQCT/O3_UMN/KInel/case.01/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

### Initialize PyCOMET model
Working case

In [ ]:
case_dir = WORKSPACE + '/ML/PyCOMET/run/SurQCT/mimmo/O3_UMN/KInel/case.01/'
sys.path.append(os.path.abspath(case_dir))

In [ ]:
os.chdir(case_dir)

Import additional libraries

In [ ]:
from pycomet.utilities import utils
from pycomet.apps import functions as fun
# Inputs
import inputs
from inputs.general import flags

Compile model

In [ ]:
# Load network architecture
net = fun._net(
    input_mod=inputs
)
# Finalize model
for k in ('train', 'save'):
    flags[k] = 0
flags['load'] = 1
model = fun._model(
    flags=flags,
    input_mod=inputs,
    net=net
)

In [ ]:
os.chdir(CWD)

### Initialize SurQCT parameters
SurQCT model inputs

In [ ]:
surqct_inp = case_dir + '/data_gen/InputData.py'
sys.path.append(os.path.dirname(surqct_inp))

import InputData

### Writing rates
Import `RatesType` module

In [ ]:
RatesType = __import__(
    f'surqct.src.RatesType.{InputData.NbAtoms}Atoms.{InputData.RatesType}',
    globals(),
    locals(),
    [InputData.RatesType],
    0
)

In [ ]:
InputData.TTranVecTest = [1500., 5000., 10000., 20000.]
InputData.MinRateValue = 1.e-16

### Writing rates

In [ ]:
for T in InputData.TTranVecTest:
    RatesType.write_predictiondata(InputData, model.net, out_dir, T)